# Vector Maxwell fields and a short trajectory

Define the local and global equations yourself, then use the generic hybrid
assembly to advance the fields. The time integration consists of free functions
in an example helper. PyMHM receives `Equation` and `LocalEquations` records;
the DG mass, curl and tangential pairing kernels are shared operations.

The physical patch uses two adjacent tetrahedral macrocells, independent vector
P2 DG fields on refined tetrahedra, and tangential P1 macroface traces. Its exact
stationary fields are $E=(1.2,-0.3,0.7)$ and $H=(0.4,0.8,-0.2)$. Four steps use
$\Delta t=0.001$. Electric and magnetic times differ by half a step.

This patch checks the declared curl, tangential orientation, boundary convention
and cross-time energy balance. It does not establish temporal convergence or an
H(curl)-conforming reconstruction of the broken DG fields.

In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "src/pymhm").is_dir()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Declare the time-discrete mathematical problem

The shared differential matrix is $C=-\mathrm{curl}$, its electric pairing is
$-C^\top$, and $Q$ includes the signed tangential face maps. The canonical trace
is $\lambda=H\times n$. Exterior data satisfy $E_{\mathrm{tan}}-\lambda=g$.
Here $Z$ is the impedance face mass and $b$ contains the prescribed moments.

For an electric kick of duration $\tau$, use its average field as the local
unknown. The initialization uses $\tau=\Delta t/2$; later kicks use
$\tau=\Delta t$.

$$
\begin{aligned}
 w &= \tfrac12(E_{\mathrm{old}}+E_{\mathrm{new}}),\\
 M_e w+\tfrac{\tau}{2}Q\lambda
   &=M_e E_{\mathrm{old}}+\tfrac{\tau}{2}(F-C^\top H),\\
 -\sum_K Q_K^\top w_K+Z\lambda&=-b,\\
 E_{\mathrm{new}}&=2w-E_{\mathrm{old}}.
\end{aligned}
$$

The magnetic stage uses an independent mass problem with no trace coordinates.
It is assembled and reconstructed by the same generic core.

$$
 M_h H_{\mathrm{new}}=M_h H_{\mathrm{old}}+\Delta t\,C E.
$$

In [ ]:
from pymhm.core.equations import Equation, LocalEquations
from pymhm.meshes.tetrahedron import TetraMesh
from threadpoolctl import threadpool_limits
from examples.tutorial_maxwell_equations import (
    Discretization, ElectricItem, MagneticItem,
    prepare, initialize, advance, l2_errors, compare_state,
)


def electric_local(item: ElectricItem) -> LocalEquations:
    """Declare the local midpoint field and its canonical tangential coupling."""
    local, Eold, H, F, duration = item
    return LocalEquations(
        a=local.electric_mass,
        L=local.electric_mass @ Eold + duration / 2 * (F - local.curl.T @ H),
        b=duration / 2 * local.coupling,
        c=-local.coupling.T,
        dofs=local.trace_dofs,
    )


def magnetic_local(item: MagneticItem) -> LocalEquations:
    """Declare the magnetic mass update with no global trace unknown."""
    local, E, Hold, dt = item
    return LocalEquations(
        a=local.magnetic_mass,
        L=local.magnetic_mass @ Hold + dt * (local.curl @ E),
        b=0, c=0, dofs=[],
    )


def electric_global(data: Discretization, boundary: np.ndarray) -> Equation:
    """Declare exterior impedance and prescribed tangential moments."""
    return Equation(a=data.impedance, L=-boundary)

## Assemble the declared spaces and physical data

`prepare` tabulates the shared curl and mass kernels; it does not select a
physical solver. The three providers above are passed to the integration
functions and consumed by `MultiscaleProblem` at each stage. Volume mass
projections initialize both fields, and the initial electric field is projected
onto the explicitly constrained tangential moments.

In [ ]:
cube = TetraMesh.unit_cube()
mesh = TetraMesh(cube.points, cube.cells[:2])
electric = np.array([1.2, -0.3, 0.7])
magnetic = np.array([0.4, 0.8, -0.2])


def impedance_data(time: float, points: np.ndarray, normals: np.ndarray) -> np.ndarray:
    """Supply the stationary physical E_tan-(H cross n) boundary data."""
    return electric - np.cross(magnetic, normals)


with threadpool_limits(1):
    data = prepare(
        mesh, time_step=0.001, degree=2, local_refinement=2,
        absorbing=1.0, boundary_data=impedance_data, quadrature_order=5,
    )
    state = initialize(
        data, electric, magnetic,
        electric_provider=electric_local, global_provider=electric_global,
    )
print({
    "dt_times_frequency_bound": data.time_step * data.frequency_bound,
    "CFL_upper_limit": 2.0,
    "electric_time": state.electric_time,
    "magnetic_time": state.magnetic_time,
})

## Advance four steps and inspect separate physical checks

The initialization places $E$ at $\Delta t/2$ and $H$ at zero. Each subsequent
step updates $H$ first and then solves the electric midpoint equations. The
reported energy uses the same cross-time convention as this leapfrog scheme.
Tangential moment defects and original electric rows are checked separately
from physical field errors.

In [ ]:
trajectory = [state]
rows = []
with threadpool_limits(1):
    for step in range(4):
        state = advance(
            data, state,
            electric_provider=electric_local,
            magnetic_provider=magnetic_local,
            global_provider=electric_global,
        )
        trajectory.append(state)
        electric_error, magnetic_error = l2_errors(data, state, electric, magnetic)
        assert max(electric_error, magnetic_error) < 1e-9
        assert abs(state.energy_balance_residual) < 1e-10
        rows.append({
            "step": step + 1,
            "electric_time": state.electric_time,
            "magnetic_time": state.magnetic_time,
            "electric_l2": electric_error,
            "magnetic_l2": magnetic_error,
            "tangential_moment_norm": state.constraint_moment_norm,
            "original_electric_residual": state.original_electric_residual,
            "energy_balance_residual": state.energy_balance_residual,
        })
for row in rows:
    print(row)

## Compare explicitly with the compatibility stepper

The primary trajectory above uses the user providers. This separate comparison
runs `MaxwellStepper` on identical geometry, coefficients, spaces, boundary data
and times. Both native factor lifetimes and numerical checks remain explicit.
Coefficient agreement is checked at initialization and after every step, along
with field norms, staggered times and energy.

In [ ]:
from pymhm._legacy.models.waves.maxwell import MaxwellStepper

comparisons = []
with threadpool_limits(1), MaxwellStepper(
    mesh, time_step=0.001, degree=2, local_refinement=2,
    absorbing=1.0, boundary_data=impedance_data, quadrature_order=5,
) as comparator:
    references = [comparator.initialize(electric, magnetic)]
    for _ in range(4):
        references.append(comparator.advance())
    for own, reference in zip(trajectory, references, strict=True):
        difference = compare_state(own, reference)
        assert difference["electric_coefficient_linf"] < 2e-12
        assert difference["magnetic_coefficient_linf"] < 2e-12
        assert difference["trace_coefficient_linf"] < 2e-11
        assert difference["energy_difference"] < 2e-14
        assert difference["electric_time_difference"] == 0.0
        assert difference["magnetic_time_difference"] == 0.0
        reference_errors = reference.l2_errors(electric, magnetic, order=5)
        assert max(reference_errors) < 1e-9
        assert abs(reference.energy_balance_residual) < 1e-10
        comparisons.append(difference)
for step, comparison in enumerate(comparisons):
    print({"step": step, **comparison})

## Retain the original analytical tutorial controls

The earlier compatibility helper still checks the same four-step patch. A zero
field control exercises homogeneous data separately; its primary equations use
the same providers and trace orientation.

In [ ]:
with threadpool_limits(1):
    zero_data = prepare(mesh, boundary_data=0.0)
    zero_state = initialize(
        zero_data, np.zeros(3), np.zeros(3),
        electric_provider=electric_local, global_provider=electric_global,
    )
    for _ in range(4):
        zero_state = advance(
            zero_data, zero_state,
            electric_provider=electric_local,
            magnetic_provider=magnetic_local,
            global_provider=electric_global,
        )
    assert l2_errors(zero_data, zero_state, np.zeros(3), np.zeros(3)) == (0.0, 0.0)

from examples.tutorial_vector_variants import run_variant
print(run_variant("maxwell-vector-3d"))
print(run_variant("maxwell-vector-3d", homogeneous=True))